### Import libraries

In [1]:
import pandas as pd

### Load the dataset

In [2]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
pcnt = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [3]:
products = products.merge(pcnt, on="product_category_name")

### Clean column names

In [4]:
def clean_columns(df):
    df.columns = df.columns.str.lower().str.strip()
    return df

orders = clean_columns(orders)
order_items = clean_columns(order_items)
customers = clean_columns(customers)
products = clean_columns(products)
order_payments = clean_columns(order_payments)
order_reviews = clean_columns(order_reviews)
sellers = clean_columns(sellers)
geolocation = clean_columns(geolocation)
pcnt = clean_columns(pcnt)

### Change date datatype to pandas data datatype

In [5]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"])
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"])

### Rename columns

In [6]:
products = products.rename(columns={"product_category_name_english": "product_category", "product_category_name": "product_category_brazilian"})

order_reviews = order_reviews.rename(
    columns={"review_comment_title": "review_title",
             "review_comment_message": "review_message"})

### Text standardization

In [7]:
products["product_category"] = products["product_category"].str.replace("_", " ")
products["product_category"] = products["product_category"].str.title()

order_payments["payment_type"] = order_payments["payment_type"].str.replace("_", " ")
order_payments["payment_type"] = order_payments["payment_type"].str.lower().str.title()


sellers["seller_city"] = sellers["seller_city"].str.title()

geolocation["geolocation_city"] = geolocation["geolocation_city"].str.title()

customers["customer_city"] = customers["customer_city"].str.title()

### Fill na

In [8]:
order_reviews["review_title"] = order_reviews["review_title"].fillna("unknown")
order_reviews["review_message"] = order_reviews["review_message"].fillna("unknown")

### Create date feautures

In [9]:
orders["year"] = orders["order_delivered_customer_date"].dt.year
orders["month"] = orders["order_delivered_customer_date"].dt.month
orders["day"] = orders["order_delivered_customer_date"].dt.day
orders["weekday"] = orders["order_delivered_customer_date"].dt.day_name()
orders["month"] = orders["order_delivered_customer_date"].dt.month_name()

### Remove duplicates

In [10]:
order_reviews["order_id"].duplicated().sum()
order_reviews = order_reviews.drop_duplicates(subset=["order_id"])

### Geoloction cleaning

In [21]:
# Check for duplicate entries
geolocation["geolocation_city"].duplicated().sum()

992152

In [22]:
# Group the table by the zip code prefix, get the mean of the lat and long, get first city and state.
geo_clean = (
    geolocation
      .groupby("geolocation_zip_code_prefix", as_index=False)
      .agg({
          "geolocation_lat": "mean",
          "geolocation_lng": "mean",
          "geolocation_city": "first",
          "geolocation_state": "first"
      })
)
geo_clean

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,Sao Paulo,SP
1,1002,-23.548146,-46.634979,Sao Paulo,SP
2,1003,-23.548994,-46.635731,Sao Paulo,SP
3,1004,-23.549799,-46.634757,Sao Paulo,SP
4,1005,-23.549456,-46.636733,Sao Paulo,SP
...,...,...,...,...,...
19010,99960,-27.953722,-52.025511,Charrua,RS
19011,99965,-28.183372,-52.039850,Agua Santa,RS
19012,99970,-28.343766,-51.874689,Ciriaco,RS
19013,99980,-28.389129,-51.843836,David Canabarro,RS


In [23]:
# Recheck for duplicates again
geo_clean["geolocation_zip_code_prefix"].duplicated().sum()

0

In [24]:
# Inspect the table
geo_clean.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,Sao Paulo,SP
1,1002,-23.548146,-46.634979,Sao Paulo,SP
2,1003,-23.548994,-46.635731,Sao Paulo,SP
3,1004,-23.549799,-46.634757,Sao Paulo,SP
4,1005,-23.549456,-46.636733,Sao Paulo,SP


In [30]:
print(f"Raw Table Shape :{geolocation.shape}\nCleaned Table Shape : {geo_clean.shape}")

Raw Table Shape :(1000163, 5)
Cleaned Table Shape : (19015, 5)


### Save to file

In [11]:
orders.to_csv("../data/cleaned/orders.csv", index=False)
order_items.to_csv("../data/cleaned/order_items.csv", index=False)
order_payments.to_csv("../data/cleaned/order_payments.csv", index=False)
order_reviews.to_csv("../data/cleaned/order_reviews.csv", index=False)
customers.to_csv("../data/cleaned/customers.csv", index=False)
products.to_csv("../data/cleaned/products.csv", index=False)
sellers.to_csv("../data/cleaned/sellers.csv", index=False)
geo_clean.to_csv("../data/cleaned/geolocation.csv", index=False)